<a href="https://colab.research.google.com/github/Amdzak/PCVK-Tugas-Kelompok/blob/main/Kelompok_Machine_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Download dataset melalui kaggle

proses download tidak menggunakan koneksi internet probadi melainkan menggunakan sisi server collab jadi jangan khawatir dengan ukuran dataset yang bsear

In [ ]:
!pip install kaggle
from google.colab import userdata
import os

try:
  os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
  os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
except Exception as e:
  print(e)

!kaggle datasets download -d moltean/fruits



Dataset URL: https://www.kaggle.com/datasets/moltean/fruits
License(s): CC-BY-SA-4.0
100% 4.52G/4.54G [01:15<00:00, 52.3MB/s]
100% 4.54G/4.54G [01:15<00:00, 64.7MB/s]


Extract dataset

In [ ]:
import zipfile
from tqdm import tqdm

zip_path = "fruits.zip"
extract_path = "fruits360"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    total_files = len(zip_ref.infolist())
    with tqdm(total=total_files, desc='Extracting') as pbar:
        for file in zip_ref.infolist():
            zip_ref.extract(file, extract_path)
            pbar.update(1)


Extracting: 100%|██████████| 280456/280456 [01:38<00:00, 2841.38it/s]


# **Proses Pra Pengolahan Data & Labeling**

## Pemetaan Ulang Label (Label Remapping): Memetakan 131 kelas asli (sub-folder) ke dalam 6 kategori target baru yang telah ditentukan

In [ ]:
import os
import shutil

source_train = "/content/fruits360/fruits-360_100x100/fruits-360/Training"
source_test = "/content/fruits360/fruits-360_100x100/fruits-360/Test"

target_base = "./Remapped_Training/Kacang-Sitrus-Melon"

# Mapping kategori → nama folder buah
category_map = {
    "Kacang-Sitrus-Melon": [
        # Kacang
        "Caju seed", "Chestnut", "Cocos", "Hazelnut",
        "Nut", "Nut Forest", "Nut Pecan", "Pistachio", "Walnut",

        # Sitrus
        "Clementine", "Grapefruit Pink", "Grapefruit White",
        "Kumquats", "Lemon", "Lemon Meyer", "Limes",
        "Mandarine", "Orange", "Pomelo Sweetie", "Tangelo",

        # Melon
        "Cantaloupe", "Melon Piel de Sapo"
    ]
}

def find_fruit_class(folder):
    for fruit in category_map["Kacang-Sitrus-Melon"]:
        if folder.startswith(fruit):
            return fruit
    return None

# Membuat folder train/test
train_target = os.path.join(target_base, "Train")
test_target  = os.path.join(target_base, "Test")
os.makedirs(train_target, exist_ok=True)
os.makedirs(test_target, exist_ok=True)

# --- PROSES TRAIN ---
for folder in os.listdir(source_train):
    fruit = find_fruit_class(folder)
    if fruit is None:
        continue

    src_folder = os.path.join(source_train, folder)
    dst_folder = os.path.join(train_target, fruit)
    os.makedirs(dst_folder, exist_ok=True)

    for img in os.listdir(src_folder):
        shutil.copy2(os.path.join(src_folder, img), os.path.join(dst_folder, img))

# --- PROSES TEST ---
for folder in os.listdir(source_test):
    fruit = find_fruit_class(folder)
    if fruit is None:
        continue

    src_folder = os.path.join(source_test, folder)
    dst_folder = os.path.join(test_target, fruit)
    os.makedirs(dst_folder, exist_ok=True)

    for img in os.listdir(src_folder):
        shutil.copy2(os.path.join(src_folder, img), os.path.join(dst_folder, img))

print("DONE! Struktur Train/Test per buah berhasil dibuat.")


DONE! Struktur Train/Test per buah berhasil dibuat.


## Scaling/Normalisasi: Mengubah rentang nilai piksel (misal: dari 0-255 menjadi 0-1) agar seragam.

In [ ]:
import os
import cv2
import numpy as np

dataset_dir = "/content/Remapped_Training/Kacang-Sitrus-Melon/Train"

images = []
labels = []

# loop setiap folder kelas
for class_name in os.listdir(dataset_dir):
    class_path = os.path.join(dataset_dir, class_name)

    # kalau bukan folder, skip
    if not os.path.isdir(class_path):
        continue

    # loop setiap gambar dalam folder kelas
    for file in os.listdir(class_path):
        img_path = os.path.join(class_path, file)

        img = cv2.imread(img_path)
        if img is None:
            continue

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # NORMALISASI
        img = img.astype("float32") / 255.0

        images.append(img)
        labels.append(class_name)

X = np.array(images)
y = np.array(labels)

print("Shape data :", X.shape)
print("Min :", X.min())
print("Max :", X.max())

Shape data : (10425, 100, 100, 3)
Min : 0.0
Max : 1.0


Hasil Output sebelum dan sesudah melakukan scaling/normalisasi

In [ ]:
# ambil 1 contoh gambar dari folder untuk BEFORE
sample_class = os.listdir(dataset_dir)[0]
sample_folder = os.path.join(dataset_dir, sample_class)
sample_file = os.listdir(sample_folder)[0]
sample_path = os.path.join(sample_folder, sample_file)

# BEFORE normalisasi
before_img = cv2.imread(sample_path)
before_img = cv2.cvtColor(before_img, cv2.COLOR_BGR2RGB)
before_img = cv2.resize(before_img, (100, 100))

print("Contoh nilai piksel sebelum Normalisasi (0–255):")
print(before_img[0:2, 0:3])

print("\nContoh nilai piksel setelah Normalisasi (0–1):")
print(img[0:2, 0:3])


Contoh nilai piksel sebelum Normalisasi (0–255):
[[[255 255 255]
  [255 255 255]
  [255 255 255]]

 [[255 255 255]
  [255 255 255]
  [255 255 255]]]

Contoh nilai piksel setelah Normalisasi (0–1):
[[[1.         0.99607843 0.98039216]
  [1.         0.99607843 0.98039216]
  [1.         1.         0.99215686]]

 [[1.         0.99607843 0.98039216]
  [0.99607843 1.         0.98039216]
  [1.         1.         0.99215686]]]


# **Proses Ekstraksi Fitur**

## Wajib mengekstraksi fitur yang relevan dari citra. Menggunakan piksel mentah (100x100x3) tidak disarankan untuk model non-DL.

### Ekstraksi Fitur Warna (HSV)

In [ ]:
# === EKSTRAKSI FITUR WARNA HSV HISTOGRAM ===

def extract_hsv_histogram(image, bins=16):
    # convert RGB -> HSV
    hsv = cv2.cvtColor((image * 255).astype("uint8"), cv2.COLOR_RGB2HSV)

    # histogram untuk masing-masing channel
    hist_h = cv2.calcHist([hsv], [0], None, [bins], [0, 180])
    hist_s = cv2.calcHist([hsv], [1], None, [bins], [0, 256])
    hist_v = cv2.calcHist([hsv], [2], None, [bins], [0, 256])

    # normalisasi histogram
    hist_h = cv2.normalize(hist_h, hist_h).flatten()
    hist_s = cv2.normalize(hist_s, hist_s).flatten()
    hist_v = cv2.normalize(hist_v, hist_v).flatten()

    # gabungkan semua histogram
    feature_vector = np.hstack([hist_h, hist_s, hist_v])

    return feature_vector


# proses ekstraksi fitur dari X (yang sudah kamu normalisasi)
X_color_features = []

for img in X:
    fitur = extract_hsv_histogram(img)
    X_color_features.append(fitur)

X_color_features = np.array(X_color_features)

print("HSV Features shape:", X_color_features.shape)


HSV Features shape: (10425, 48)


### Ekstraksi Fitur Tekstur (LBP)

In [ ]:
from skimage.feature import local_binary_pattern
import numpy as np
import cv2

# fungsi LBP (aman)
def extract_lbp_features(gray_img_uint8, radius=3):
    n_points = 8 * radius

    # LBP harus pakai uint8 → aman, tanpa warning
    lbp = local_binary_pattern(gray_img_uint8, n_points, radius, method='uniform')

    # histogram LBP
    hist, _ = np.histogram(
        lbp.ravel(),
        bins=np.arange(0, n_points + 3),
        range=(0, n_points + 2)
    )

    # normalisasi histogram (ini aman)
    hist = hist.astype("float32")
    hist /= hist.sum()

    return hist

# === EKSTRAKSI FITUR TEKSTUR DARI X ===
X_texture_features = []

for img in X:
    # convert RGB normalized (0–1) → RGB 0–255
    img_uint8 = (img * 255).astype('uint8')

    # convert ke grayscale uint8 (no warning)
    gray_uint8 = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)

    # ekstraksi fitur tekstur LBP
    fitur_lbp = extract_lbp_features(gray_uint8)
    X_texture_features.append(fitur_lbp)

X_texture_features = np.array(X_texture_features)

print("LBP Features shape:", X_texture_features.shape)


LBP Features shape: (10425, 26)


# **Reduksi Dimensi**

## Disarankan melakukan reduksi dimensi (misal: PCA - Principal Component Analysis) setelah ekstraksi fitur untuk mengurangi beban komputasi.

menggabungkan fitur menjadi 1

In [ ]:
# gabungkan fitur warna + tekstur
X_combined = np.hstack([X_color_features, X_texture_features])

print("Combined feature shape:", X_combined.shape)


Combined feature shape: (10425, 74)


Menentukan n_components yang diperlukan untuk Menjaga variansi data sebanyak mungkin sambil mengurangi dimensi.

In [ ]:
from sklearn.decomposition import PCA
import numpy as np

pca_test = PCA().fit(X_combined)

explained = np.cumsum(pca_test.explained_variance_ratio_)

optimal = np.argmax(explained >= 0.99) + 1

# print 50 komponen pertama
for i in range(1, 51):
    print(f"Komponen {i}: {explained[i-1]*100:.2f}% variansi tercapture")


Komponen 1: 29.83% variansi tercapture
Komponen 2: 51.38% variansi tercapture
Komponen 3: 61.67% variansi tercapture
Komponen 4: 69.96% variansi tercapture
Komponen 5: 75.34% variansi tercapture
Komponen 6: 79.16% variansi tercapture
Komponen 7: 82.64% variansi tercapture
Komponen 8: 85.63% variansi tercapture
Komponen 9: 88.31% variansi tercapture
Komponen 10: 90.34% variansi tercapture
Komponen 11: 92.07% variansi tercapture
Komponen 12: 93.45% variansi tercapture
Komponen 13: 94.37% variansi tercapture
Komponen 14: 95.18% variansi tercapture
Komponen 15: 95.85% variansi tercapture
Komponen 16: 96.44% variansi tercapture
Komponen 17: 96.91% variansi tercapture
Komponen 18: 97.34% variansi tercapture
Komponen 19: 97.73% variansi tercapture
Komponen 20: 98.11% variansi tercapture
Komponen 21: 98.36% variansi tercapture
Komponen 22: 98.57% variansi tercapture
Komponen 23: 98.77% variansi tercapture
Komponen 24: 98.94% variansi tercapture
Komponen 25: 99.09% variansi tercapture
Komponen 

menggunakan n_component = 46 karena udah mencover 99.99% data penting

In [ ]:
from sklearn.decomposition import PCA

# pakai PCA dengan 46 komponen
pca = PCA(n_components=46)

# fit + transform dataset gabungan (warna + tekstur)
X_pca = pca.fit_transform(X_combined)

print("Shape PCA:", X_pca.shape)

# cek variansi yang diambil oleh 46 komponen
print("Total Explained Variance:", pca.explained_variance_ratio_.sum())


Shape PCA: (10425, 46)
Total Explained Variance: 0.9998663


# Pembuatan Data Latih dan Data Uji

# Pembuatan Model

# Evaluasi Model